# 習題：數學物件、Python 型別與 AI 中的數值表示

在數學中，我們經常接觸下列概念：

- 自然數（Natural Numbers）
- 整數（Integers）
- 有理數（Rational Numbers）
- 實數（Real Numbers）
- 複數（Complex Numbers）
- 向量（Vectors）
- 矩陣（Matrices）
- 集合（Sets）
- 命題的真假（Truth Values）

本習題將探討：

```text
數學物件
    ↓
Python 型別
    ↓
CPython 實作
    ↓
AI 中的數值表示
```

---

## 第一部分：Python 如何表示數學物件？

請查閱 Python 文件或自行測試，完成下表。

| 數學概念 | Python 型別 | 範例 |
|----------|------------|------|
| 自然數 | int | 5 |
| 整數 | int | -5 |
| 有理數 | fractions.Fraction | Fraction(1, 3) |
| 實數 | float | 3.14159 |
| 複數 | complex | 3 + 4j |
| 向量 | list / tuple | [1.0, 2.0, 3.0] |
| 矩陣 | list | [[1, 2], [3, 4]] |
| 集合 | set（可變）/ frozenset（不可變） | {1, 2, 3} |
| 命題真假 |bool | True / False |

並回答下列問題：

1. 哪些是 Python 的內建型別（Built-in Types）？ int、float、complex、list、tuple、set、frozenset、bool
2. 哪些需要額外模組才能較完整地表示？ 有理數：需引用標準模組 fractions.Fraction。

                                   高精度小數：需引用標準模組 decimal.Decimal。

                                   高效能向量與矩陣：需要第三方套件（如 numpy、PyTorch、TensorFlow 中的數值陣列/張量型別）。
3. Python 的型別分類與數學中的分類是否完全一致？ 並不完全一致。數學上的數系具嚴格的包含關係（如 $\mathbb{N} \subseteq \mathbb{Z} \subseteq \mathbb{Q} \subseteq \mathbb{R} \subseteq \mathbb{C}$），但 Python 是基於「計算機記憶體佈局」與「性能效率」設計。例如自然數與負整數統一用 int 儲存；實數在 Python 中預設以符合 IEEE 754 標準的雙精度浮點數 float 表示，無法包含所有無限不循環的實數（無理數）。

---


## 第二部分：數學概念與 Python 型別

數學上有下列包含關係：

$$
\mathbb N
\subseteq
\mathbb Z
\subseteq
\mathbb Q
\subseteq
\mathbb R
\subseteq
\mathbb C
$$

請回答：

1. Python 是否為上述每一個數系都提供獨立型別？ 沒有。Python 沒有為自然數（$\mathbb{N}$）與有理數（$\mathbb{Q}$）提供獨立的內建型別。
2. 為什麼自然數與整數都使用 `int` 表示？ 在電腦底層，處理包含正負號的整數運算邏輯是統一且高效的，為自然數獨立設立型別會增加語法與型別檢查的複雜度。
3. Python 為什麼提供 `complex` 型別？ 複數在科學計算、訊號處理、物理模擬與工程數學中被廣泛應用。Python 原生提供 complex 型別，能讓語法直接支援 3 + 4j 等運算，不需手動綁定兩個 float 實部與虛部。
4. Python 為什麼沒有內建的有理數型別？ 大多數常規程式計算使用 IEEE 754 浮點數（float）即可滿足需求且硬體計算速度極快。有理數需要分子與分母兩個整數，每次加減乘除均需計算最大公因數（GCD）進行約分，效能開銷較大，因此被放進標準庫 fractions 模組供特殊情境調用。
5. 下列三個物件的型別有何不同？ 1 的型別是 int（整數）。

1.0 的型別是 float（IEEE 754 雙精度浮點數）。

1+0j 的型別是 complex（複數，由實部 1.0 與虛部 0.0 兩個雙精度浮點數組成）。

```python
1
1.0
1+0j
```

---

## 第三部分：CPython 如何實作這些型別？

除了知道 Python 使用哪些型別表示數學物件之外，

我們也希望了解：

> Python 物件在 CPython 中是如何實作的？

請查閱資料，完成下表。

| 數學概念 | Python 型別 | CPython 實作 | 主要特點 |
|----------|------------|-------------|----------|
| 自然數 | `int` | PyLongObject | 變長陣列儲存位元組（Digit array），可動態擴充，實現任意精度（Arbitrary Precision）。 |
| 整數 | `int` | PyLongObject | 變長陣列儲存位元組（Digit array），可動態擴充，實現任意精度（Arbitrary Precision） |
| 實數 | `float` | PyFloatObject | 包裝 C 語言的 double |
| 複數 | `complex` | PyComplexObject | 內嵌 Py_complex 結構，包含兩個 C 雙精度浮點數 |
| 集合 | `set` | PySetObject | 雜湊表（Hash Table）開放尋址法，元素須可雜湊 |
| 命題真假 | `bool` | PyBoolObject |PyLongObject 的子類別，全域僅有 Py_True 與 Py_False 兩個單例 |
| 串列 | `list` | PyListObject | 動態陣列，儲存指向各 Python 物件指標的陣列 |

---

## 第四部分：深入探討各種型別

請針對第三部分中的每一種型別回答：

1. 它是否為 Python 的內建型別？ 是
2. 對應的 CPython 物件結構名稱為何？ PyLongObject
3. 它如何儲存資料？ 標頭檔加上動態長度陣列
4. 是否有固定大小限制？ 無固定位元數限制
5. 若資料規模持續增加，如何擴充儲存空間？ 當數值超出當前陣列長度時，動態重新分配更多空間

---

## 第五部分：Python 整數與浮點數

執行下列程式：

```python
print(10**100)
print(10**1000)
```

回答：

1. 為什麼 Python 能表示如此大的整數？ Python 3 的整數不是採用硬體固定的 32 位元或 64 位元暫存器儲存，而是使用動態記憶體陣列（PyLongObject）來切割並儲存大數。
2. Python 的 `int` 是否具有固定的位元數限制？ 沒有硬性位元限制
3. 什麼是 Arbitrary Precision（任意精度）？ 指數字的表示精度僅受限於電腦當前可用的記憶體總量，而非由硬體的暫存器大小決定
4. CPython 使用什麼物件來實作 `int`？ PyLongObject
5. Python 整數大小真正的限制是什麼？ 系統可用的記憶體空間（RAM）以及 Python 對字串轉換等的保護限制

---

再執行：

```python
0.1 + 0.2
```

```python
0.1 + 0.2 == 0.3
```

```python
format(0.1, ".20f")
```

回答：

1. 為什麼結果與數學上的答案不同？ 十進位小數（如 0.1 與 0.2）轉成二進位時會變成無限循環小數。電腦只能截斷儲存，導致產生微小截斷誤差（Truncation Error），實際計算結果為 0.30000000000000004。
2. 浮點數是否能精確表示所有小數？ 無法精確表示所有十進位小數
3. 什麼是浮點數的精度（precision）？ 指機能有效區分的最少有效位數
4. 什麼是浮點數的範圍（range）？ 指浮點數能表示的最大與最小數值區間
5. 精度與範圍有何不同？ 精度（Precision）：決定數值「準不準」（有效數字的位數）。

範圍（Range）：決定數字「大不大」或「小不小」（指數部分的上下界）。

---

## 第六部分：AI 模型中的數值表示

人工智慧模型需要儲存大量參數（Parameters）與權重（Weights）。

請查閱資料並回答：

1. AI 模型中的權重通常使用什麼型別儲存？ 主要使用低精度的浮點數格式，如 FP32、FP16、BF16 或 INT8/INT4 量化格式
2. 什麼是 FP64？ 64 位元雙精度浮點數（1 位元符號、11 位元指數、52 位元尾數）。精度極高，但記憶體與計算開銷極大。
3. 什麼是 FP32？ 32 位元單精度浮點數（1 位元符號、8 位元指數、23 位元尾數）。傳統深度學習的標準格式。
4. 什麼是 FP16？ 16 位元半精度浮點數（1 位元符號、5 位元指數、10 位元尾數）。節省記憶體，但指數範圍較小，容易發生下溢（Underflow）或溢位（Overflow）。
5. 什麼是 BF16？ Brain Floating Point 16 位元格式（1 位元符號、8 位元指數、7 位元尾數）。維持與 FP32 相同的指數範圍（8 位元），大幅降低溢位風險，特別適合大型語言模型（LLM）訓練。
6. 為什麼大型 AI 模型不直接使用 Python 的 `int` 作為權重？ PyLongObject 為高階封裝物件，包含大量 Python Header 雜項開銷，無法直接被 GPU 向量化平行計算。

權重矩陣運算需要連續的記憶體區塊與硬體級矩陣乘法器（Tensor Cores）加速。

AI 模型中的梯度下降法需要連續空間的微積分導數，非離散的整數。

---

## 第七部分：比較不同數值表示方式

完成下表。

| 表示方式 | 位元數 | 是否有誤差 | 主要用途 |
|----------|--------|------------|----------|
| int | 動態 | 無誤差 | 常規程式邏輯、計數器、不容許誤差的整數運算 |
| FP64 | 64 位元 | 有誤差 | 高精度科學計算、物理模擬、金融工程 |
| FP32 | 32 位元 | 有誤差 | 傳統 AI 模型訓練、圖形渲染、科學計算 |
| FP16 | 16 位元 | 有誤差 | AI 推論、混合精度訓練 |
| BF16 | 16 位元 | 有誤差 | 現代大型 AI 模型（如 LLM）訓練與推論 |

並回答：

1. 哪種表示方式最精確？ Python int（整數完全無誤差）；浮點數中則為 FP64
2. 哪種表示方式最節省記憶體？ 在給出的選項中為 FP16 與 BF16
3. 哪種表示方式最常見於 AI 訓練？ BF16 與 FP32
4. 哪種表示方式最常見於 AI 推論（Inference）？ FP16 以及更低精度的量化格式

---

## 第八部分：綜合思考

### 思考題一

Python 官方文件指出：

```python
int
```

具有 Unlimited Precision（無限制精度）。

請回答：

1. Unlimited Precision 是什麼意思？ 指 Python 的整數長度不受 32 位元或 64 位元暫存器限制，計算結果永遠精確，不會發生傳統硬體整數溢位（Overflow）環繞的問題
2. 它是否代表可以儲存無限大的整數？ 否。理論上無上限，但實際上受限於電腦的實體記憶體（RAM）容量與處理時間
3. CPython 如何實現這項特性？ 採用 PyLongObject 結構，將大整數拆解成多個小區塊（digits），儲存在動態陣列中，並重寫相關的加減乘除算法（如 Karatsuba 乘法）

---

### 思考題二

比較：

```python
int
```

與

```python
float
```

回答：

1. 為什麼 `int` 幾乎沒有固定上限？ int 是軟體層面動態分配陣列儲存的，長度可隨數字增加而擴充
2. 為什麼 `float` 卻存在範圍限制？ float 是直接使用硬體層面的 IEEE 754 雙精度 64 位元格式，記憶體大小固定，指數位元（11 位元）決定了它的上限與下限。
3. 為什麼 `float` 可能產生誤差？ 二進位無法完全精確表示所有十進位小數（例如十進位的 0.1 在二進位中是無限循環小數），且受限於 52 位元尾數長度，多餘位元會被截斷捨入。

---

### 思考題三

比較下列兩種設計理念：

#### Python 的整數

```text
Arbitrary Precision
正確性優先
```

#### AI 模型常用的 FP16、BF16

```text
Limited Precision
效率優先
```

請說明：

1. 兩種設計理念的差異。 Python 整數（正確性優先）：保證數值計算完全精確，採用動態陣列與高階物件，不惜犠牲運算速度與記憶體空間。

AI 模型 FP16/BF16（效率優先）：為了追求極致的計算速度與空間壓縮，甘願放棄部分尾數精度，換取極高的矩陣運算吞吐量。
2. 各自的優點與缺點。 正確性優先（Arbitrary Precision）：優點為計算精確、無溢位風險；缺點為速度慢、記憶體消耗大、無法被 GPU 並行加速。

效率優先（Limited Precision）：優點為記憶體需求低、能完全發揮 GPU/TPU 矩陣運算單元（Tensor Core）效能；缺點為存在捨入誤差與數值不穩定風險。
3. 為什麼大型語言模型通常選擇較低精度的表示方式？ 大型語言模型含有數百億至數千億個參數，低精度（16 位元）可將記憶體容量需求減半，大幅節省 GPU VRAM 成本；同時 GPU 對 16 位元浮點數的運算吞吐量（TFLOPS）遠高於 32 位元或 64 位元。研究也表明，神經網路具有高容錯性（Noise Robustness），微小的精度損失不影響最終的生成效果。
4. 如果所有 AI 模型都改用 FP64，可能會產生哪些問題？ VRAM 顯存爆炸：記憶體需求變為 BF16 的 4 倍，單張 GPU 將無法載入模型。

計算效能極低：現代 GPU 的 Tensor Cores 主要針對 FP16/BF16/FP8 優化，FP64 的計算速度會慢上數倍至數十倍。

頻寬瓶頸：記憶體頻寬（Memory Bandwidth）會成為嚴重瓶頸，導致模型推論與訓練極其緩慢。

---

### 挑戰題

請用自己的話解釋：

```python
x = 10**1000
```

為什麼 Python 可以正常運作；

而大型 AI 模型卻通常選擇使用：

```text
FP32
FP16
BF16
```

而不是任意精度整數或高精度浮點數？

請從：

- 記憶體使用量
- 計算速度
- 數值精度
- GPU 運算效率

等角度進行討論。

在 Python 中執行 x = 10**1000 時，Python 使用了軟體層級的 PyLongObject。這是一個動態陣列，會將這個巨大的數字拆解成許多個小段（Digit）並保存在 RAM 中。因為這僅涉及單一純量的標量運算，CPU 只需要花費微秒級的時間分配記憶體並計算即可完成，因此 Python 能正常運作。   相反地，大型 AI 模型不採用任意精度整數或高精度浮點數（如 FP64），而選擇 FP32、FP16 或 BF16，原因如下：   記憶體使用量（VRAM）：一個 700 億參數（70B）的模型，若使用 FP64 儲存需要 $700 \times 8 = 560\text{ GB}$ 的顯存；若換成 BF16/FP16 則僅需 $700 \times 2 = 140\text{ GB}$，降低了 75% 的顯存需求。   計算速度與 GPU 運算效率：現代 GPU（如 NVIDIA H100/A100）內建專門為矩陣乘法設計的 Tensor Cores。Tensor Cores 處理 FP16/BF16 的硬體吞吐量（TFLOPS）通常是 FP64 的數十倍。任意精度整數（Arbitrary Precision）無法被編譯成硬體級別的固定管道指令，完全無法在 GPU 上進行大規模平行計算。   數值精度容忍度（Neural Network Fault Tolerance）：神經網路本質上是高度備份且抗噪的統計模型，模型輸出對權重中極微小的小數變化並不敏感。使用 BF16 雖損失了部分尾數精度，但其指數範圍與 FP32 相同，既能保證梯度不溢位，又不會影響模型最終的學習品質。   綜上所述，Python int 選擇了「單點計算的絕對正確」，而 AI 模型選擇了「大規模矩陣運算的極致吞吐與效率」。 